# Softmax
*Converting raw scores into probabilities — the universal normalizer of deep learning.*

---
**Component:** `learning/kernels/softmax/`
**Companion kernels:** `v0_naive.py` → `sm89_v1_block_per_row.py` → `sm89_v2_smem_tree_reduce.py` → `sm89_v3_online_single_pass.py`


## What Is Softmax?

**In plain English:** Softmax takes a list of arbitrary numbers ("scores" or "logits") and converts them into a probability distribution — a list of non-negative numbers that sum to exactly 1.

Think of it as a **voting machine**: each score gets exponentially more votes the higher it is, then all votes are divided by the total so you end up with percentages.

**Where it appears in a transformer:**
- **Attention weights** — "how much does token A pay attention to token B?"
- **Output head** — "what is the probability of each next word in the vocabulary?"


In [ ]:
import math

# We'll trace through everything by hand, then compare to library versions
print("Setup complete — numpy and math imported")


## Worked Example: Step by Step

**Input (logits):**

$$x = [2.0,\ 1.0,\ 0.1]$$

| Step | Operation | Result |
|------|-----------|--------|
| 1 | $e^{2.0}$ | 7.389 |
| 1 | $e^{1.0}$ | 2.718 |
| 1 | $e^{0.1}$ | 1.105 |
| 2 | Sum: $7.389 + 2.718 + 1.105$ | 11.212 |
| 3 | $7.389 / 11.212$ | **0.659** |
| 3 | $2.718 / 11.212$ | **0.242** |
| 3 | $1.105 / 11.212$ | **0.099** |

**Output:** $p = [0.659,\ 0.242,\ 0.099]$

✅ Check: $0.659 + 0.242 + 0.099 = 1.000$
✅ Highest input (2.0) → highest probability (0.659)
✅ All values in (0, 1)


In [ ]:
x = [2.0, 1.0, 0.1]

print("Step 1 — exponentiate each element:")
exps = [math.exp(xi) for xi in x]
for xi, ei in zip(x, exps):
    print(f"  exp({xi}) = {ei:.3f}")

total = sum(exps)
print(f"\nStep 2 — sum: {' + '.join(f'{e:.3f}' for e in exps)} = {total:.3f}")

p = [e / total for e in exps]
print(f"\nStep 3 — divide each by {total:.3f}:")
for ei, pi in zip(exps, p):
    print(f"  {ei:.3f} / {total:.3f} = {pi:.3f}")

print(f"\nOutput: {[round(pi, 3) for pi in p]}")
print(f"Sum check: {sum(p):.10f}")


## The Formula

$$\boxed{\text{softmax}(x)_i = \frac{e^{x_i}}{\displaystyle\sum_{j=1}^{n} e^{x_j}}}$$

### Symbol Definitions

| Symbol | Name | Meaning in our example |
|--------|------|----------------------|
| $x_i$ | logit at index $i$ | Raw score, e.g. $x_0 = 2.0$ |
| $e$ | Euler's number | $\approx 2.718$; makes outputs positive |
| $e^{x_i}$ | exponential of $x_i$ | Always $> 0$; grows fast with $x_i$ |
| $n$ | length of vector | $n = 3$ in our example |
| $\sum_{j=1}^{n}$ | sum over all $j$ | Ensures the denominator = total votes |
| $\text{softmax}(x)_i$ | output probability $i$ | $\in (0,1)$, all outputs sum to 1 |

### 🗣️ Say It Out Loud
> *"The softmax of vector x at position i equals: e to the power of x-sub-i, divided by the sum — for all j from 1 to n — of e to the power of x-sub-j."*

**Tutor's gloss:** "We're turning raw scores into a competition. The `exp` makes every score positive and amplifies differences — a score of 2 gets $e^2 = 7.4$ votes while a score of 1 gets only $e^1 = 2.7$ votes. Dividing by the total converts absolute vote counts into percentages."


## The Stability Problem — and the Fix

### Why Naive Softmax Breaks

Try $x = [1000.0,\ 1001.0]$:

$$e^{1000} \approx 5 \times 10^{434} \quad \text{(float32 max is } \approx 3.4 \times 10^{38}\text{)}$$

Result: `inf / inf = NaN` — training crashes.

### The Stability Fix: Subtract the Maximum

Let $m = \max_j x_j$. Then:

$$\text{softmax}(x)_i = \frac{e^{x_i - m}}{\displaystyle\sum_j e^{x_j - m}}$$

### 🗣️ Why This Is Mathematically Identical

$$\frac{e^{x_i - m}}{\sum_j e^{x_j - m}} = \frac{e^{x_i} \cdot e^{-m}}{\sum_j e^{x_j} \cdot e^{-m}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

The $e^{-m}$ **cancels top and bottom** — the result is unchanged.
After shifting, every exponent $\leq 0$, so $e^{x_i - m} \in (0, 1]$. No overflow possible.


In [ ]:
def softmax_naive(x):
    exps = [math.exp(xi) for xi in x]
    total = sum(exps)
    return [e / total for e in exps]

def softmax_stable(x):
    m = max(x)
    exps = [math.exp(xi - m) for xi in x]
    total = sum(exps)
    return [e / total for e in exps]

# Overflow test
x_large = [1000.0, 1001.0]
print("Naive on [1000, 1001]:")
try:
    result = softmax_naive(x_large)
    nan_inf = any(not (0 < v < 1) for v in result)
    print(f"  {result}  {'← BAD (NaN/inf)' if nan_inf else 'ok'}")
except (OverflowError, ZeroDivisionError) as e:
    print(f"  CRASHED: {e}")

print("\nStable on [1000, 1001]:")
result = softmax_stable(x_large)
print(f"  {[round(v, 4) for v in result]}  ← correct!")
assert abs(sum(result) - 1.0) < 1e-9


## Online Softmax: One Pass Instead of Two

**Normal 2-pass approach:**
1. Scan all elements → find max $m$
2. Scan again → compute $e^{x_j - m}$ and sum

**Online approach (Milakov & Gimelshein, 2018):** Fuse both passes into one scan by maintaining a running state $(m, d)$:

- $m$ = maximum element seen **so far**
- $d$ = sum of $e^{x_j - m}$ for all $j$ seen **so far**

### Update Rule (for each new element $x_i$):

$$m_\text{new} = \max(m_\text{old},\ x_i)$$

$$d_\text{new} = d_\text{old} \cdot e^{m_\text{old} - m_\text{new}} + e^{x_i - m_\text{new}}$$

### 🗣️ Why the Rescaling Factor?

> *"When we see a new maximum, the old running sum $d$ was computed relative to the old max. We multiply by $e^{m_\text{old} - m_\text{new}}$ to re-express it relative to the new max. If no new max is found, $m_\text{new} = m_\text{old}$, so $e^0 = 1$ and $d$ is just accumulated unchanged."*

### Key Invariant After Seeing $x_0, \ldots, x_{k-1}$:
$$m = \max(x_0, \ldots, x_{k-1}), \quad d = \sum_{j < k} e^{x_j - m}$$


In [ ]:
def softmax_online(x):
    m = float('-inf')
    d = 0.0
    for xi in x:
        m_new = max(m, xi)
        rescale = math.exp(m - m_new) if m != float('-inf') else 0.0
        d = d * rescale + math.exp(xi - m_new)
        m = m_new
    return [math.exp(xi - m) / d for xi in x]

# Trace through x = [2.0, 1.0, 0.1]
x = [2.0, 1.0, 0.1]
m, d = float('-inf'), 0.0
print(f"Start:  m = -∞,  d = 0.0\n")
for xi in x:
    m_new = max(m, xi)
    rescale = math.exp(m - m_new) if m != float('-inf') else 0.0
    d_new = d * rescale + math.exp(xi - m_new)
    print(f"See {xi:4.1f}:  m_new = {m_new:.1f},  "
          f"rescale = {rescale:.3f},  d_new = {d_new:.3f}")
    m, d = m_new, d_new

result = [math.exp(xi - m) / d for xi in x]
print(f"\nFinal: m={m}, d={d:.3f}")
print(f"Output: {[round(v, 3) for v in result]}")
assert all(abs(a - b) < 1e-9 for a, b in zip(result, softmax_stable(x)))
print("✓ Matches stable 2-pass result")


## Parallel Merge Operator (for GPU Tree Reduction)

When 128 threads each process a chunk of the row, every thread produces its own local state $(m_t, d_t)$. To combine them into one global $(m, d)$:

$$\text{merge}\bigl((m_1, d_1),\ (m_2, d_2)\bigr) = \Bigl(\max(m_1, m_2),\ d_1 \cdot e^{m_1 - m} + d_2 \cdot e^{m_2 - m}\Bigr)$$

where $m = \max(m_1, m_2)$.

This operator is **associative**: `merge(merge(A,B), C) == merge(A, merge(B,C))`
→ a binary tree works; $\log_2(128) = 7$ rounds suffice for 128 threads.

**This is the same operation used in FlashAttention** to merge partial attention outputs computed on separate KV tiles.

### Log-Sum-Exp Connection

$$\text{LSE}(x) = \log\!\sum_j e^{x_j} = m + \log(d)$$

FlashAttention stores LSE per row (not the full softmax) so the backward pass can recompute $e^{x_i - \text{LSE}} = \text{softmax}(x)_i$ without materializing the $T \times T$ attention matrix.


## Optimization Ladder

| Version | Threads/row | HBM passes | Key idea |
|---------|-------------|------------|----------|
| `v0_naive` | 1 | 3 | One thread does everything serially |
| `sm89_v1_block_per_row` | 128 | 3 | 128 threads split columns; tree reduce in SMEM |
| `sm89_v2_smem_tree_reduce` | 128 | 3 | Better SMEM layout; explicit tree reduction |
| `sm89_v3_online_single_pass` | 128 | **2** | Online fuses pass-1+pass-2; 33% less HBM reads |

**Why memory bandwidth dominates:**
Softmax does ≈5 FLOPs per element but reads/writes 8 bytes.
Arithmetic intensity ≈ 0.6 FLOP/byte vs RTX 4080 ridge point ≈ 230 FLOP/byte.
→ Memory is the bottleneck, not compute. Reducing passes = the win.


## CuTeDSL: Online Softmax in One Pass

The v3 kernel (`sm89_v3_online_single_pass`) fuses the max-finding and sum-finding passes
into a single scan. Each thread tracks a tiny running state — just two numbers — instead of
storing intermediate values.

```python
@cute.kernel
def softmax_online_kernel(mIn, mOut):
    row = blockIdx.x    # one CTA per row
    tid = threadIdx.x   # 0 to 127

    # Each thread tracks: the max it has seen, and the exp-sum relative to that max
    m_local = float('-inf')
    d_local = 0.0

    # One pass over the row — each element is read exactly once
    for col in cutlass.range(tid, N, 128):
        xi = mIn[row, col]

        m_new = max(m_local, xi)
        # When the max rises, all previous exp() terms need rescaling.
        # exp(xi - m_old) = exp(xi - m_new) * exp(m_old - m_new)
        # So multiply the old sum by exp(m_old - m_new), then add the new term.
        d_local = d_local * cute.exp(m_local - m_new) + cute.exp(xi - m_new)
        m_local = m_new

    # Merge 128 local (m, d) states into one global (m, d) via shared memory
    smem_m[tid] = m_local
    smem_d[tid] = d_local
    cute.sync_threads()

    for half in [64, 32, 16, 8, 4, 2, 1]:
        if tid < half:
            m1, d1 = smem_m[tid],        smem_d[tid]
            m2, d2 = smem_m[tid + half], smem_d[tid + half]
            m_new = max(m1, m2)
            smem_m[tid] = m_new
            smem_d[tid] = d1 * cute.exp(m1 - m_new) + d2 * cute.exp(m2 - m_new)
        cute.sync_threads()

    global_m = smem_m[0]
    global_d = smem_d[0]

    # Second pass — can't avoid this, but it's the only second pass
    for col in cutlass.range(tid, N, 128):
        mOut[row, col] = cute.exp(mIn[row, col] - global_m) / global_d
```

**The rescaling line unpacked:**
`d_local = d_local * exp(m_local - m_new) + exp(xi - m_new)`

Think of `d_local` as a sum that was correct when the running max was `m_local`.
A new element `xi` arrives with a higher value `m_new`.
Every previous exp term was computed as `exp(something - m_local)`.
To stay consistent, each must become `exp(something - m_new)`.
That's the same as multiplying each by `exp(m_local - m_new)`.
So we multiply the whole old sum by that factor, then add the new term.
No values are stored — only `m_local` and `d_local` live in registers.

## RTX 4080: Why Passes Matter More Than FLOPs

Softmax does ~5 FLOPs per element against 4 bytes (2 read + 2 write) in BF16.
Intensity: ~1.25 FLOP/byte — still well below the 151 F/B ridge. **Memory-bound.**

The v0 naive kernel makes 3 HBM passes (find max, compute sum, normalize).
The v3 online kernel makes 2 passes (fused scan + normalize pass).
That 33% reduction in HBM reads translates directly to 33% faster kernel time.

**No further architectural tricks help here** — you can't do softmax in one total pass because you need the final (m, d) before you can write any output. Two is the minimum.

For attention specifically: the benefit compounds with FlashAttention. Flash uses online softmax so it never writes the full [T × T] score matrix to HBM. For T=4096: saving 4096 × 4096 × 4 bytes = 64 MB per head per layer from being written and re-read.

## Warp Shuffle: Better Reduction for Short Attention Rows

The v2 kernel uses a shared-memory tree reduce. That's appropriate for long rows
(e.g., vocabulary softmax with N=150,000). But attention softmax rows are short —
N is the sequence length, typically 128 to 4096 — and warp shuffles are faster.

**At N=128:** 128 elements ÷ 128 threads = 1 element per thread.
The entire reduction fits inside one warp shuffle. Zero SMEM, five rounds.

**At N=4096:** 4096 ÷ 128 threads = 32 elements per thread.
Each of the 4 warps runs its own 5-round shuffle reduce.
Then only the 4 warp-level results go into SMEM for one final pass.
Total SMEM traffic: 4 writes + 4 reads, vs 128 writes + 128 reads in the tree approach.

**Warp-shuffle merge for two (m, d) states:**
```python
for offset in [16, 8, 4, 2, 1]:
    m_other = cute.shfl_xor(m_lane, offset)   # get the paired lane's max
    d_other = cute.shfl_xor(d_lane, offset)   # get the paired lane's denominator
    m_new   = max(m_lane, m_other)
    d_lane  = d_lane * exp(m_lane - m_new) + d_other * exp(m_other - m_new)
    m_lane  = m_new
# After 5 rounds: lane 0 holds the warp's global (m, d)
```

**Why faster:** Each shuffle instruction takes ~4 cycles and touches no memory.
SMEM loads have ~20+ cycle latency. At N=4096, the hybrid approach trades
128 SMEM accesses for 4, saving ~120 × 20 = 2,400 cycles of latency per softmax row.